<p style="align: center;"><img src="https://static.tildacdn.com/tild6636-3531-4239-b465-376364646465/Deep_Learning_School.png" width="400"></p>

# Глубокое обучение. Часть 2
# Домашнее задание по теме "Механизм внимания"

Это домашнее задание проходит в формате peer-review. Это означает, что его будут проверять ваши однокурсники. Поэтому пишите разборчивый код, добавляйте комментарии и пишите выводы после проделанной работы.

В этом задании вы будете решать задачу классификации математических задач по темам (многоклассовая классификация) с помощью Transformer.

В качестве датасета возьмем датасет математических задач по разным темам. Нам необходим следующий файл:

[Файл с классами](https://docs.google.com/spreadsheets/d/13YIbphbWc62sfa-bCh8MLQWKizaXbQK9/edit?usp=drive_link&ouid=104379615679964018037&rtpof=true&sd=true)

**Hint:** не перезаписывайте модели, которые вы получите на каждом из этапов этого дз. Они ещё понадобятся.

### Задание 1 (2 балла)

Напишите кастомный класс для модели трансформера для задачи классификации, использующей в качествке backbone какую-то из моделей huggingface.

Т.е. конструктор класса должен принимать на вход название модели и подгружать её из huggingface, а затем использовать в качестве backbone (достаточно возможности использовать в качестве backbone те модели, которые упомянуты в последующих пунктах)

In [6]:
import torch
import torch.nn as nn
from typing import Union
from transformers import AutoModel

### This is just an interface example. You may change it if you want.
import torch
import torch.nn as nn
from typing import Union
from transformers import AutoModel

class TransformerClassificationModel(nn.Module):
    def __init__(self, base_transformer_model: Union[str, nn.Module],
                 num_classes: int):
        super().__init__()
        # выберем модель из Hugging Face - base_transformer_model и загружаем ее
        # именно backbone
        self.backbone = AutoModel.from_pretrained(base_transformer_model)
        # create additional layers for classfication
        # hidden_size размер скрытого вектора
        hidden_size = self.backbone.config.hidden_size
        # Создаем слой
        # num_classes количество классов, которые предсказывает модель
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, inputs):
        # propagate inputs through the model. Return dict with logits
        # токенизированный текст пропускается через трансформер, мы получаем его выходы
        backbone_outputs = self.backbone(**inputs)
        # представление текста передается в классификатор и мы получаем оценки классов
        cls_embedding = backbone_outputs.last_hidden_state[:, 0, :]
        # смотрим на вектор первого токена (он представляет весь текст)
        logits = self.classifier(cls_embedding)

        outputs = {"logits": logits}
        return outputs

### Задание 2 (1 балл)

Напишите функцию заморозки backbone у модели (если необходимо, возвращайте из функции модель)

In [7]:
def freeze_backbone_function(model: TransformerClassificationModel):
  # проходим по всем параметрам backbone
  for param in model.backbone.parameters():
    # не считаем градиенты
    param.requires_grad = False
  # вернем модель с замороженным backbone
  return model

### Задание 3 (2 балла)

Напишите функцию, которая будет использована для тренировки (дообучения) трансформера (TransformerClassificationModel). Функция должна поддерживать обучение с замороженным и размороженным backbone.

In [8]:
import copy

def train_transformer(transformer_model, num_epochs = 3, lr=2e-5, freeze_backbone=True):
    model = copy.copy(transformer_model)
    ### YOUR CODE IS HERE
    for param in model.backbone.parameters():
        param.requires_grad = not freeze_backbone

    # оптимизатор создаем только для обучаемых параметров
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    # функция потерь для классификации
    criterion = nn.CrossEntropyLoss()

    # переводим модель в режим обучения
    model.train()

    # обучение
    for epoch in range(num_epochs):
        for batch in train_dataloader:
            inputs = {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"]
            }
            labels = batch["labels"]

            # обнуляем старые градиенты
            optimizer.zero_grad()

            # logits
            outputs = model(inputs)
            logits = outputs["logits"]

            #  loss
            loss = criterion(logits, labels)

            # считаем градиенты
            loss.backward()

            # обновляем параметры
            optimizer.step()

    finetuned_model = model

    return finetuned_model

### Задание 4 (1 балл)

Проверьте вашу функцию из предыдущего пункта, дообучив двумя способами
*cointegrated/rubert-tiny2* из huggingface.

In [11]:
import pandas as pd

df = pd.read_excel("/content/data_problems_translated.xlsx")
print(df.columns)


Index(['Unnamed: 0', 'problem_text', 'topic'], dtype='object')


In [12]:
print(df.head())

   Unnamed: 0                                       problem_text  \
0           0  To prove that the sum of the numbers of the ex...   
1           1  ( b) Will the statement of the previous challe...   
2           2  The quadratic three-member graph with the coef...   
3           3  Can you draw on the surface of Rubik's cube a ...   
4           4  Dima, who came from Vrunlandia, said that ther...   

           topic  
0  number_theory  
1  number_theory  
2       polynoms  
3  combinatorics  
4         graphs  


In [13]:
num_classes = df["topic"].nunique()
print(num_classes)

7


In [14]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

label2id = {label: i for i, label in enumerate(df["topic"].unique())}
df["label"] = df["topic"].map(label2id)
num_classes = df["label"].nunique()
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)
tokenizer = AutoTokenizer.from_pretrained("cointegrated/rubert-tiny2")

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = TextDataset(train_df["problem_text"], train_df["label"])
test_dataset = TextDataset(test_df["problem_text"], test_df["label"])

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [17]:
rubert_tiny_transformer_model = TransformerClassificationModel(
    "cointegrated/rubert-tiny2", num_classes)
rubert_tiny_finetuned_with_freezed_backbone = train_transformer(rubert_tiny_transformer_model, freeze_backbone=True)

rubert_tiny_transformer_model = TransformerClassificationModel(
    "cointegrated/rubert-tiny2", num_classes)
rubert_tiny_full_finetuned = train_transformer(rubert_tiny_transformer_model, freeze_backbone=False)

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def evaluate_model(model, test_dataloader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_dataloader:
            inputs = {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"]
            }
            labels = batch["labels"]

            outputs = model(inputs)
            logits = outputs["logits"]
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [22]:
acc_frozen = evaluate_model(rubert_tiny_finetuned_with_freezed_backbone, test_dataloader)
acc_full = evaluate_model(rubert_tiny_full_finetuned, test_dataloader)

print(f"с замороженным backbone {acc_frozen}")
print(f"с размороженным backbone {acc_full}")

с замороженным backbone 0.46824644549763034
с размороженным backbone 0.6037914691943128


### Задание 5 (1 балл)

Обучите *tbs17/MathBert* (с замороженным backbone и без заморозки), проанализируйте результаты. Сравните скоры с первым заданием. Получилось лучше или нет? Почему?

In [16]:
tokenizer = AutoTokenizer.from_pretrained("tbs17/MathBERT")

train_dataset = TextDataset(train_df["problem_text"], train_df["label"])
test_dataset = TextDataset(test_df["problem_text"], test_df["label"])

train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [23]:
rubert_tiny_base = TransformerClassificationModel(
    "cointegrated/rubert-tiny2", num_classes
)

mathbert_base = TransformerClassificationModel(
    "tbs17/MathBERT", num_classes
)

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: tbs17/MathBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Задание 6 (1 балл)

Напишите функцию для отрисовки карт внимания первого слоя для моделей из задания

In [30]:
from typing import List
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

def draw_first_layer_attention_maps(attention_head_ids: List[int], text: str, model):
    model.eval()

    model_name = model.backbone.config._name_or_path
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model.backbone(
            **encoding,
            output_attentions=True,
            return_dict=True
        )

    if outputs.attentions is None:
        raise ValueError("Model did not return attentions. outputs.attentions is None")

    first_layer_attention = outputs.attentions[0]
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

    for head_id in attention_head_ids:
        attention_map = first_layer_attention[0, head_id].cpu().numpy()

        plt.figure(figsize=(8, 6))
        plt.imshow(attention_map)
        plt.colorbar()
        plt.xticks(range(len(tokens)), tokens, rotation=90)
        plt.yticks(range(len(tokens)), tokens)
        plt.title(f"First layer attention, head {head_id}")
        plt.tight_layout()
        plt.show()

In [31]:

rubert_tiny_base = TransformerClassificationModel(
    "cointegrated/rubert-tiny2", num_classes
)
text = test_df.iloc[0]["problem_text"]
draw_first_layer_attention_maps([0, 1, 2], text, rubert_tiny_base)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ValueError: Model did not return attentions. outputs.attentions is None

### Задание 7 (1 балл)

Проведите инференс для всех моделей **ДО ДООБУЧЕНИЯ** на 2-3 текстах из датасета. Посмотрите на головы Attention первого слоя в каждой модели на выбранных текстах (отрисуйте их отдельно).

Попробуйте их проинтерпретировать. Какие связи улавливают карты внимания? (если в модели много голов Attention, то проинтерпретируйте наиболее интересные)

In [ ]:
### YOUR CODE IS HERE

### Задание 8 (1 балл)

Сделайте то же самое для дообученных моделей. Изменились ли карты внимания и связи, которые они улавливают? Почему?

In [ ]:
### YOUR CODE IS HERE